In [22]:
import warnings
import os
import json
from random import randrange
from functools import partial
import torch
torch.cuda.set_device(0)
from datasets import Dataset
from datasets import load_dataset
from transformers import (AutoModelForCausalLM,
                          AutoTokenizer,
                          BitsAndBytesConfig,
                          HfArgumentParser,
                          Trainer,
                          TrainingArguments,
                          DataCollatorForLanguageModeling,
                          EarlyStoppingCallback,
                          pipeline,
                          logging,
                          set_seed)

import bitsandbytes as bnb
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel, AutoPeftModelForCausalLM
from trl import SFTTrainer
import pandas as pd
from tqdm import tqdm
from utils import write_and_print

In [23]:
import pyarrow as pa
print(pa.__version__)

17.0.0


In [24]:
torch.__version__

'1.13.0'

## Settings

### Basic Settings

In [25]:
VERSION = 1
UTTERANCE_TYPE = 'Answer' # Alternate options: 'Question' or 'Answer'
LABEL = 'Party'          # Alternate options: 'Party' 'Majority'

# Load the dataset from checkpoint? Setting it to False makes the script read the
# dataframe from memory and create the train and test set again
LOAD_DATASETS = True

# What should the size of the test split be?
TRAIN_SPLIT_LENGTH = 10000
VALIDATION_SPLIT_LENGTH = 1000

# How many examples do we want to train on?
TRAIN_EXAMPLES = 50

# Are we loading the model from a check point? True or False
LOAD_MODEL = False

FINE_TUNE = True

# Failsafe
if not FINE_TUNE and not LOAD_DATASETS:
    warnings.warn('LOAD_DATASETS set to False. Train and Test sets will be created and saved again')
if not FINE_TUNE and not LOAD_MODEL:
    raise ValueError('Cannot test model without Loading model first. Please set LOAD_MODEL and LOAD_MODEL_PATH')

### Savepath Parameters

In [26]:
'''The parameters must always be set'''

# Path to the dataframe pickle file
DF_PATH = '/mnt/fstore/DataFiles/PickledFiles/Data_MARK2.pkl'

# Path to the saved Train and Test sets
SAVE_TRAINSET_PATH = '/home/manjari/CongressionalHearingsProject/QnA/Classification/Llama2/MARK2/'+ LABEL +'/'+ UTTERANCE_TYPE +'/trainSet.csv'
SAVE_VALIDATIONSET_PATH = '/home/manjari/CongressionalHearingsProject/QnA/Classification/Llama2/MARK2/'+ LABEL +'/'+ UTTERANCE_TYPE +'/validationSet.csv'
SAVE_TESTSET_PATH = '/home/manjari/CongressionalHearingsProject/QnA/Classification/Llama2/MARK2/'+ LABEL +'/'+ UTTERANCE_TYPE +'/testSet.csv'

# Output directory where the model predictions and checkpoints will be stored
SAVE_CHECKPOINT_DIR = '/home/manjari/CongressionalHearingsProject/QnA/Classification/Llama2/MARK2/'+ LABEL +'_'+ UTTERANCE_TYPE +'_Mark2.'+ str(VERSION)

#Output JSON file path for the model predictions
SAVE_OUTPUT_PATH = '/home/manjari/CongressionalHearingsProject/QnA/Classification/Llama2/MARK2/'+ LABEL +'/'+ UTTERANCE_TYPE +f'/output_{VERSION}'

# Path to the log file
LOG_PATH = '/home/manjari/CongressionalHearingsProject/QnA/Classification/Llama2/MARK2/Llama2_Answer_Party.log'
write_and_print(f'Beginning version {VERSION}\n========================\n',LOG_PATH)

Beginning version 1



### Loadpath Parameters

In [27]:
'''These parameters need to be set the LOAD attributes are set'''

# Path for loading the saved model
LOAD_CHECKPOINT_DIR = '/home/manjari/CongressionalHearingsProject/QnA/Classification/Llama2/MARK2/final_merged_checkpoint'  

# Path for loading saved datasets
LOAD_TRAINSET_PATH = '/home/manjari/CongressionalHearingsProject/QnA/Classification/Llama2/MARK2/Party/Answer/trainSet.csv'
LOAD_VALIDATIONSET_PATH = '/home/manjari/CongressionalHearingsProject/QnA/Classification/Llama2/MARK2/Party/Answer/validationSet.csv'
LOAD_TESTSET_PATH = '/home/manjari/CongressionalHearingsProject/QnA/Classification/Llama2/MARK2/Party/Answer/testSet.csv'

### Training Arguments and Parameters

In [28]:
# Batch size per GPU for training
per_device_train_batch_size = 1

# Number of update steps to accumulate the gradients for
gradient_accumulation_steps = 5

# Initial learning rate (AdamW optimizer)
learning_rate = 2e-4

# Optimizer to use
optim = "paged_adamw_32bit"

# Number of training steps (overrides num_train_epochs)
max_steps = 50

# Linear warmup steps from 0 to learning_rate
warmup_steps = 5

# Enable fp16/bf16 training (set bf16 to True with an A100)
fp16 = True

# Log every X updates steps
logging_steps = 1

### Transformer Parameters

In [29]:
# The pre-trained model from the Hugging Face Hub to load and fine-tune
model_name = "daryl149/llama-2-7b-chat-hf"

### bitsandbytes Parameters

In [30]:
# Activate 4-bit precision base model loading
load_in_4bit = True

# Activate nested quantization for 4-bit base models (double quantization)
bnb_4bit_use_double_quant = True

# Quantization type (fp4 or nf4)
bnb_4bit_quant_type = "nf4"

### QLoRA Parameters

In [31]:
# LoRA attention dimension
lora_r = 16

# Alpha parameter for LoRA scaling
lora_alpha = 64

# Dropout probability for LoRA layers
lora_dropout = 0.1

# Bias
bias = "none"

# Task type
task_type = "CAUSAL_LM"

## Setting up the Model

In [47]:
def create_bnb_config(load_in_4bit, bnb_4bit_use_double_quant, bnb_4bit_quant_type, bnb_4bit_compute_dtype):
    """
    Configures model quantization method using bitsandbytes to speed up training and inference

    :param load_in_4bit: Load model in 4-bit precision mode
    :param bnb_4bit_use_double_quant: Nested quantization for 4-bit model
    :param bnb_4bit_quant_type: Quantization data type for 4-bit model
    :param bnb_4bit_compute_dtype: Computation data type for 4-bit model
    """
    
    bnb_config = BitsAndBytesConfig(
        load_in_4bit = load_in_4bit,
        bnb_4bit_use_double_quant = bnb_4bit_use_double_quant,
        bnb_4bit_quant_type = bnb_4bit_quant_type,
        bnb_4bit_compute_dtype = bnb_4bit_compute_dtype,
    )

    return bnb_config

def load_model(model_name, bnb_config):
    """
    Loads model and model tokenizer

    :param model_name: Hugging Face model name
    :param bnb_config: Bitsandbytes configuration
    """

    # Get number of GPU device and set maximum memory
    n_gpus = 1 # torch.cuda.device_count()
    max_memory = f'{40960}MB'

    # Load model
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config = bnb_config,
        device_map = "sequential", # dispatch the model efficiently on the available resources
        max_memory = {i: max_memory for i in range(n_gpus)},
    )

    # Load model tokenizer with the user authentication token
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token = False)

    # Set padding token as EOS token
    tokenizer.pad_token = tokenizer.eos_token

    return model, tokenizer

## Setting up the Datastes

In [48]:
def create_prompt_formats(sample):
    """
    Creates a formatted prompt template for a prompt in the instruction dataset

    :param sample: Prompt or sample from the instruction dataset
    """

    # Initialize static strings for the prompt template
    INTRO_BLURB = "You are an AI model that analyzes text to determine political context."
    INSTRUCTION_KEY = "### Instruction: "
    INSTRUCTION_BLURB = "Below is a transcript of a spoken response. Based on the text identify whether the speaker was responding to a question from a Democrat or a Republican. Respond with \"R\" for Republican or \"D\" for Democrat. Do not explain."
    INPUT_KEY = "Input: "
    RESPONSE_KEY = "### Response: "
    END_KEY = "### End"

    # Combine a prompt with the static strings
    blurb = f"{INTRO_BLURB}"
    instruction = f"{INSTRUCTION_KEY} {INSTRUCTION_BLURB}"
    input_context = f"{INPUT_KEY}{sample['Utterance']}" if sample["Utterance"] else None
    response = f"{RESPONSE_KEY}{sample['Party']}"
    end = f"{END_KEY}"

    # Create a list of prompt template elements
    parts = [part for part in [blurb, instruction, input_context, response, end] if part]

    # Join prompt template elements into a single string to create the prompt template
    formatted_prompt = "\n".join(parts)

    # Store the formatted prompt template in a new key "text"
    sample["text"] = formatted_prompt

    return sample

def create_prompt_formats_test(sample):
    """
    Creates a formatted prompt template for a prompt in the instruction dataset

    :param sample: Prompt or sample from the instruction dataset
    """

    # Initialize static strings for the prompt template
    INTRO_BLURB = "You are an AI model that analyzes text to determine political context."
    INSTRUCTION_KEY = "### Instruction: "
    INSTRUCTION_BLURB = "Below is a transcript of a spoken response. Based on the text identify whether the speaker was responding to a question from a Democrat or a Republican. Respond with \"R\" for Republican or \"D\" for Democrat. Do not explain."
    INPUT_KEY = "Input: "
    END_KEY = "### End"

    # Combine a prompt with the static strings
    blurb = f"{INTRO_BLURB}"
    instruction = f"{INSTRUCTION_KEY} {INSTRUCTION_BLURB}"
    input_context = f"{INPUT_KEY}{sample['Utterance']}" if sample["Utterance"] else None
    end = f"{END_KEY}"

    # Create a list of prompt template elements
    parts = [part for part in [blurb, instruction, input_context, end] if part]

    # Join prompt template elements into a single string to create the prompt template
    formatted_prompt = "\n".join(parts)

    # Store the formatted prompt template in a new key "text"
    sample["text"] = formatted_prompt

    return sample

def get_max_length(model):
    """
    Extracts maximum token length from the model configuration

    :param model: Hugging Face model
    """

    # Pull model configuration
    conf = model.config
    # Initialize a "max_length" variable to store maximum sequence length as null
    max_length = None
    # Find maximum sequence length in the model configuration and save it in "max_length" if found
    for length_setting in ["n_positions", "max_position_embeddings", "seq_length"]:
        max_length = getattr(model.config, length_setting, None)
        if max_length:
            # write_and_print(f"Found max lenth: {max_length - 30}", LOG_PATH)
            break
    # Set "max_length" to 1024 (default value) if maximum sequence length is not found in the model configuration
    if not max_length:
        max_length = 1024
        print(f"Using default max length: {max_length}")
    return max_length - 30

def preprocess_batch(batch, tokenizer, max_length):
    """
    Tokenizes dataset batch

    :param batch: Dataset batch
    :param tokenizer: Model tokenizer
    :param max_length: Maximum number of tokens to emit from the tokenizer
    """

    return tokenizer(
        batch["text"],
        max_length = max_length,
        truncation = True,
    )

def preprocess_dataset(tokenizer: AutoTokenizer, max_length: int, seed, dataset: str, isTest = False):
    """
    Tokenizes dataset for fine-tuning

    :param tokenizer (AutoTokenizer): Model tokenizer
    :param max_length (int): Maximum number of tokens to emit from the tokenizer
    :param seed: Random seed for reproducibility
    :param dataset (str): Instruction dataset
    """

    # Add prompt to each sample
    write_and_print("Preprocessing dataset...", LOG_PATH)
    if isTest:
        dataset = dataset.map(create_prompt_formats_test)
    else:
        dataset = dataset.map(create_prompt_formats)

    # Apply preprocessing to each batch of the dataset & and remove input and text fields
    _preprocessing_function = partial(preprocess_batch, max_length = max_length, tokenizer = tokenizer)
    dataset = dataset.map(
        _preprocessing_function,
        batched = True,
        remove_columns = ["Utterance", "text"],
    )

    # Filter out samples that have "input_ids" exceeding "max_length"
    dataset = dataset.filter(lambda sample: len(sample["input_ids"]) < max_length)

    return dataset

## Setting up the Fine Tuning Task

In [49]:
def create_peft_config(r, lora_alpha, target_modules, lora_dropout, bias, task_type):
    """
    Creates Parameter-Efficient Fine-Tuning configuration for the model

    :param r: LoRA attention dimension
    :param lora_alpha: Alpha parameter for LoRA scaling
    :param modules: Names of the modules to apply LoRA to
    :param lora_dropout: Dropout Probability for LoRA layers
    :param bias: Specifies if the bias parameters should be trained
    """
    config = LoraConfig(
        r = r,
        lora_alpha = lora_alpha,
        target_modules = target_modules,
        lora_dropout = lora_dropout,
        bias = bias,
        task_type = task_type,
    )

    return config

def find_all_linear_names(model):
    """
    Find modules to apply LoRA to.

    :param model: PEFT model
    """

    cls = bnb.nn.Linear4bit
    lora_module_names = set()
    for name, module in model.named_modules():
        if isinstance(module, cls):
            names = name.split('.')
            lora_module_names.add(names[0] if len(names) == 1 else names[-1])

    if 'lm_head' in lora_module_names:
        lora_module_names.remove('lm_head')
    write_and_print(f"LoRA module names: {list(lora_module_names)}", LOG_PATH)
    return list(lora_module_names)

def print_trainable_parameters(model, use_4bit = False):
    """
    Prints the number of trainable parameters in the model.

    :param model: PEFT model
    """

    trainable_params = 0
    all_param = 0

    for _, param in model.named_parameters():
        num_params = param.numel()
        if num_params == 0 and hasattr(param, "ds_numel"):
            num_params = param.ds_numel
        all_param += num_params
        if param.requires_grad:
            trainable_params += num_params

    if use_4bit:
        trainable_params /= 2

    write_and_print(
        f"All Parameters: {all_param:,d} || Trainable Parameters: {trainable_params:,d} || Trainable Parameters %: {100 * trainable_params / all_param}", LOG_PATH)
    
def fine_tune(model,
          tokenizer,
          dataset,
          lora_r,
          lora_alpha,
          lora_dropout,
          bias,
          task_type,
          per_device_train_batch_size,
          gradient_accumulation_steps,
          warmup_steps,
          max_steps,
          learning_rate,
          fp16,
          logging_steps,
          output_dir,
          optim):
    """
    Prepares and fine-tune the pre-trained model.

    :param model: Pre-trained Hugging Face model
    :param tokenizer: Model tokenizer
    :param dataset: Preprocessed training dataset
    """

    # Enable gradient checkpointing to reduce memory usage during fine-tuning
    model.gradient_checkpointing_enable()

    # Prepare the model for training 
    model = prepare_model_for_kbit_training(model)

    # Get LoRA module names
    target_modules = find_all_linear_names(model)

    # Create PEFT configuration for these modules and wrap the model to PEFT
    peft_config = create_peft_config(lora_r, lora_alpha, target_modules, lora_dropout, bias, task_type)
    model = get_peft_model(model, peft_config)

    # Print information about the percentage of trainable parameters
    print_trainable_parameters(model)

    # Training parameters
    trainer = Trainer(
        model = model,
        train_dataset = dataset,
        args = TrainingArguments(
            per_device_train_batch_size = per_device_train_batch_size,
            gradient_accumulation_steps = gradient_accumulation_steps,
            warmup_steps = warmup_steps,
            max_steps = max_steps,
            learning_rate = learning_rate,
            fp16 = fp16,
            logging_steps = logging_steps,
            output_dir = output_dir,
            optim = optim,
            device=0
        ),
        data_collator = DataCollatorForLanguageModeling(tokenizer, mlm = False)
    )

    model.config.use_cache = False

    do_train = True

    # Launch training and log metrics
    write_and_print("Training...", LOG_PATH)

    if do_train:
        train_result = trainer.train()
        metrics = train_result.metrics
        trainer.log_metrics("train", metrics)
        trainer.save_metrics("train", metrics)
        trainer.save_state()
        print(metrics)

    # Save model
    write_and_print("Saving last checkpoint of the model...", LOG_PATH)
    os.makedirs(output_dir, exist_ok = True)
    trainer.model.save_pretrained(output_dir)

    # Free memory for merging weights
    del model
    del trainer
    torch.cuda.empty_cache()

## Loading the model

In [50]:
if LOAD_MODEL:
    model = AutoModelForCausalLM.from_pretrained(LOAD_CHECKPOINT_DIR)
    model = model.to('cuda')
    tokenizer = AutoTokenizer.from_pretrained(LOAD_CHECKPOINT_DIR)
else:
    # Compute data type for 4-bit base models
    bnb_4bit_compute_dtype = torch.bfloat16

    # Load model from Hugging Face Hub with model name and bitsandbytes configuration
    bnb_config = create_bnb_config(load_in_4bit, bnb_4bit_use_double_quant, bnb_4bit_quant_type, bnb_4bit_compute_dtype)
    model, tokenizer = load_model(model_name, bnb_config)

PackageNotFoundError: No package metadata was found for accelerate

## Loading the Datasets

In [ ]:
df = pd.read_pickle(DF_PATH)

COLUMN_FILTER = 'is_answer'  if UTTERANCE_TYPE == 'Answer' else 'is_question'
LABEL1 = 'R' if LABEL == 'Party' else True
LABEL2 = 'D' if LABEL == 'Party' else False

df = df.loc[((df[LABEL] == LABEL1)|(df[LABEL] == LABEL2))&(df[COLUMN_FILTER] == True),['Utterance',LABEL]]

if LOAD_DATASETS:
    # TODO
    write_and_print('Loading dataset...', LOG_PATH)
    train_ds = load_dataset('csv', data_files=LOAD_TRAINSET_PATH)['train']
    val_ds = load_dataset('csv', data_files=LOAD_VALIDATIONSET_PATH)['train']
    # test_ds = load_dataset('csv', data_files=LOAD_TESTSET_PATH)['train']

    #Random seed
    seed = 33

    max_length = get_max_length(model)
    preprocessed_dataset = preprocess_dataset(tokenizer, max_length, seed, train_ds.select(list(range(TRAIN_EXAMPLES))), isTest=False)
    preprocessed_train_ds = preprocess_dataset(tokenizer, max_length, seed, train_ds.select(list(range(TRAIN_EXAMPLES))), isTest=True)
    preprocessed_val_ds = preprocess_dataset(tokenizer, max_length, seed, train_ds, isTest=True)
    # preprocessed_test_ds = preprocess_dataset(tokenizer, max_length, seed, test_ds, isTest= True)

    write_and_print(f'Preprocessed Train Dataset:\n{preprocessed_train_ds}', LOG_PATH)
    write_and_print(f'Preprocessed Train Dataset:\n{preprocessed_val_ds}', LOG_PATH)
    # write_and_print(f'Preprocessed Test Dataset:\n{preprocessed_test_ds}', LOG_PATH)

else: 
    write_and_print('Creating new dataset...', LOG_PATH)
    write_and_print(f'Length of the full dataset = {len(df)}',LOG_PATH)

    dataset = Dataset.from_pandas(df)
    my_dict = dataset.train_test_split(test_size=1-(TRAIN_SPLIT_LENGTH/len(dataset)), shuffle=True)
    train_ds = my_dict['train']
    temp_ds = my_dict['test']
    
    my_dict = temp_ds.train_test_split(test_size=1-(VALIDATION_SPLIT_LENGTH/len(temp_ds)), shuffle=True)
    val_ds = my_dict['train']
    test_ds = my_dict['test']
    
    write_and_print(f'Number of prompts in train: {len(train_ds)}', LOG_PATH)
    write_and_print(f'Column names are: {train_ds.column_names}', LOG_PATH)
    write_and_print(f'Number of prompts in test: {len(test_ds)}', LOG_PATH)
    write_and_print(f'Column names are: {test_ds.column_names}', LOG_PATH)

    write_and_print(f'Formatting the datasets sanity check: {create_prompt_formats(train_ds[randrange(len(train_ds))])}', LOG_PATH)

    # Random seed
    seed = 33

    max_length = get_max_length(model)
    preprocessed_dataset = preprocess_dataset(tokenizer, max_length, seed, train_ds.select(list(range(TRAIN_EXAMPLES))), isTest=False)
    preprocessed_train_ds = preprocess_dataset(tokenizer, max_length, seed, train_ds.select(list(range(TRAIN_EXAMPLES))), isTest=True)
    preprocessed_val_ds = preprocess_dataset(tokenizer, max_length, seed, train_ds, isTest=True)
    preprocessed_test_ds = preprocess_dataset(tokenizer, max_length, seed, test_ds, isTest= True)

    write_and_print(f'Preprocessed Train Dataset:\n{preprocessed_train_ds}', LOG_PATH)
    write_and_print(f'Preprocessed Validation Dataset:\n{preprocessed_val_ds}', LOG_PATH)
    write_and_print(f'Preprocessed Test Dataset:\n{preprocessed_test_ds}', LOG_PATH)

    train_ds.to_csv(SAVE_TRAINSET_PATH)
    val_ds.to_csv(SAVE_VALIDATIONSET_PATH)
    test_ds.to_csv(SAVE_TESTSET_PATH)

Loading dataset...
Preprocessing dataset...
Preprocessing dataset...


Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Filter:   0%|          | 0/50 [00:00<?, ? examples/s]

Preprocessing dataset...
Preprocessed Train Dataset:
Dataset({
    features: ['Party', 'conversation_id', 'input_ids', 'attention_mask'],
    num_rows: 50
})
Preprocessed Train Dataset:
Dataset({
    features: ['Party', 'conversation_id', 'input_ids', 'attention_mask'],
    num_rows: 9942
})


## Finetune

In [42]:
if FINE_TUNE:
     # Fine-tune model to perform new task
    fine_tune(model, tokenizer, preprocessed_dataset, lora_r, lora_alpha, lora_dropout, bias, task_type, per_device_train_batch_size, gradient_accumulation_steps, warmup_steps, max_steps, learning_rate, fp16, logging_steps, SAVE_CHECKPOINT_DIR, optim)

    # Load fine-tuned weights
    model = AutoPeftModelForCausalLM.from_pretrained(SAVE_CHECKPOINT_DIR, device_map = "auto", torch_dtype = torch.bfloat16)
    # Merge the LoRA layers with the base model
    model = model.merge_and_unload()

    # Save fine-tuned model at a new location
    output_merged_dir = SAVE_CHECKPOINT_DIR+"/final_merged_checkpoint"
    os.makedirs(output_merged_dir, exist_ok = True)
    model.save_pretrained(output_merged_dir, safe_serialization = True)

    # Save tokenizer for easy inference
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.save_pretrained(output_merged_dir)

LoRA module names: []


ValueError: Target modules [] not found in the base model. Please check the target modules and try again.

### Getting Accuracies

In [ ]:
def extract_response(text):
    return text[text.index('### Response: ')+14:text.index('### Response: ')+15].replace('\n','')

def contains_response(text):
    if '### Response: D' in text or '### Response: R' in text:
        return True
    else:
        return False
    
def get_accuracy(llama_responses):
    res_df = df.copy()
    total_correct = 0
    total = 0
    problem_conversation_ids = []
    for conversation_id in tqdm(llama_responses):
        # print(conversation_id)
        if contains_response(llama_responses[conversation_id]):
            predicted = extract_response(llama_responses[conversation_id])
            # print(predicted)
            res_df.loc[conversation_id,'Predicted_Party'] = predicted
            total += 1
            if predicted == res_df.loc[conversation_id,'Party']:
                total_correct += 1
        else:
            problem_conversation_ids.append(conversation_id)
            
    return total, total_correct



#### Train Accuracy

In [ ]:
outputs = {}
for sample in tqdm(preprocessed_train_ds):
    output = model.generate(torch.LongTensor([sample['input_ids']]).to('cuda'), max_length = len(sample['input_ids'])+25) 
    text = tokenizer.decode(output[0])
    outputs[sample['conversation_id']] = text
    with open(SAVE_OUTPUT_PATH+'_train.json', "w") as outfile: 
        json.dump(outputs, outfile)
total, total_corret = get_accuracy(outputs)
write_and_print(f'Train Accuracy = {total_corret/total}')

#### Validation Accuracy

In [ ]:
outputs = {}
for sample in tqdm(preprocessed_val_ds):
    output = model.generate(torch.LongTensor([sample['input_ids']]).to('cuda'), max_length = len(sample['input_ids'])+25) 
    text = tokenizer.decode(output[0])
    outputs[sample['conversation_id']] = text
    with open(SAVE_OUTPUT_PATH+'_validation.json', "w") as outfile: 
        json.dump(outputs, outfile)
total, total_corret = get_accuracy(outputs)
write_and_print(f'Validation Accuracy = {total_corret/total}')